# NCF candidate-generation baseline

A static implicit-feedback neural collaborative filtering (NCF) baseline
trained on session clicks.

- Positives are session clicks from the full `labels_train` pool; article
  history exposures are no longer treated as positives.
- The model contains two `nn.Embedding` blocks: one for users and one for
  articles. Their 64-d vectors are concatenated into a 128-d vector and passed
  through an MLP with shape `128 -> 64 -> 32 -> 1`.
- There is no dot product in the score.
- Training uses the `LOSS` objective (BPR or BCE) with up to four exposed
  session negatives redrawn for each positive every epoch.
- Validation candidates exclude the request's full causal history and use the
  eligible catalogue at the request timestamp.

The notebook runs the full pipeline on the complete training and target splits.


## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import load_context, resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
RUN_LOSSES = ("bpr", "bce")
LOSS_CONFIGS = {
    loss: {
        "model_name": f"ncf_{loss}",
        "artifact_dir": ARTIFACTS / f"ncf_{loss}",
        "recommendations_path": ARTIFACTS
        / f"ncf_{loss}"
        / f"recommendations_{TARGET_SPLIT}_top500.parquet",
        "result_path": RESULTS
        / f"candidate_generation_ncf_{loss}_{TARGET_SPLIT}.csv",
    }
    for loss in RUN_LOSSES
}
for config in LOSS_CONFIGS.values():
    config["artifact_dir"].mkdir(parents=True, exist_ok=True)
COMPARISON_PATH = RESULTS / f"candidate_generation_comparison_{TARGET_SPLIT}.csv"

M_VALUES = (50, 100, 200, 500, 750, 1000, 1500)
CANDIDATE_CAP = max(M_VALUES)
EMBED_DIM = 128
MLP_HIDDEN = (128, 64, 32)
SCORE_PAIRS = 524_288
NUM_EPOCHS = 50
BATCH_SIZE = 32_768
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-6
SEED = 4222
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Repository:", ROOT)
print("Model:", LOSS_CONFIGS[RUN_LOSSES[0]]["model_name"])
print("Device:", DEVICE)
print("Torch:", torch.__version__)


## 2. Data ingestion and model-related processing

### Session clicks and per-epoch session negatives

Positives come from the full `labels_train` pool: each row is one clicked article
from a training request, so only clicks in that session are treated as positive.
History-only exposures are no longer positives. Each training epoch redraws up to
four exposed non-clicks from the same slate. Article codes are mapped to
`article_row` through `article_index.parquet`.

In [ ]:
context = load_context(ROOT)

In [ ]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet",
    columns=["article_row", "news_code"],
).sort("article_row")

run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

train_profiles = pl.read_parquet(
    sorted((FEATURES / "request_profiles_train").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp"],
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = train_profiles.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(train_profiles["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    train_profiles.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build NCF training arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(train_profiles, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["user_code"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups,
        groups["pos_row"].to_numpy().astype(np.int64),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    train_profiles.select("impression_uid", "user_code"),
    max_impressions=None,
    seed=SEED + 1,
)
TUNE_UIDS = tune["impression_uid"].to_numpy()
TUNE_USERS = tune["user_code"].to_numpy().astype(np.int64)
TUNE_ITEMS = tune["article_row"].to_numpy().astype(np.int64)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)

print(f"Users: {context.n_users:,}")
print(f"Articles: {context.n_items:,}")
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print(f"Tuning cutoff: {np.datetime64(cutoff_ns, 'ns')}")
print(f"Tuning rows: {tune.height:,}")
print(f"Tuning NDCG k (max training clicks per impression): {TUNE_NDCG_K}")


## 3. Setting up the model

### Two embeddings, concatenation and an MLP

`NCF` has exactly two embedding tables. The user and article vectors are
concatenated into a 128-d vector and passed through `128 -> 64 -> 32 -> 1` with
ReLU activations. The scalar MLP output is the ranking score. BPR compares the
MLP score of a session-click positive against the sampled exposed non-clicks
for that positive (at most four per epoch). No false-negative filtering is applied.

In [ ]:
class NCF(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, hidden_dims):
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, embed_dim)
        self.item_embedding = nn.Embedding(n_items, embed_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.01)
        nn.init.normal_(self.item_embedding.weight, std=0.01)
        with torch.no_grad():
            self.item_embedding.weight[0].zero_()

        layers = []
        in_dim = 2 * embed_dim
        for hidden_dim in hidden_dims:
            layers += [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
            in_dim = hidden_dim
        self.mlp = nn.Sequential(*layers, nn.Linear(in_dim, 1))

    def score_embeddings(self, user_vectors, item_vectors):
        return self.mlp(torch.cat([user_vectors, item_vectors], dim=-1)).squeeze(-1)

    def forward(self, user_ids, item_ids):
        return self.score_embeddings(
            self.user_embedding(user_ids),
            self.item_embedding(item_ids),
        )


## 4. Training the model

### BPR/BCE training, chronological tuning and refit

In [ ]:
def train_epoch(model, optimizer, user_ids, item_ids, neg_matrix, neg_len, rng):
    model.train()
    order = rng.permutation(len(user_ids))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(user_ids[batch], dtype=torch.long, device=DEVICE)
        positives = torch.as_tensor(item_ids[batch], dtype=torch.long, device=DEVICE)
        negatives = torch.as_tensor(neg_matrix[batch], dtype=torch.long, device=DEVICE)
        neg_lens = torch.as_tensor(neg_len[batch], dtype=torch.long, device=DEVICE)
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        users_expanded = users[:, None].expand(-1, negatives.shape[1])
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(
            users_expanded.reshape(-1), negatives.reshape(-1)
        ).reshape(len(batch), negatives.shape[1])
        if LOSS == "bpr":
            loss = -(
                F.logsigmoid(positive_scores[:, None] - negative_scores) * neg_mask
            ).sum() / neg_mask.sum()
        else:
            loss = F.binary_cross_entropy_with_logits(
                torch.cat([positive_scores, negative_scores[neg_mask]]),
                torch.cat(
                    [
                        torch.ones_like(positive_scores),
                        torch.zeros_like(negative_scores[neg_mask]),
                    ]
                ),
            )
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [ ]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                TUNE_USERS[start:end], dtype=torch.long, device=DEVICE
            )
            items = torch.as_tensor(
                TUNE_ITEMS[start:end], dtype=torch.long, device=DEVICE
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K)
    return ndcg


In [ ]:
TRAINED_MODELS = {}
for loss in RUN_LOSSES:
    LOSS = loss
    config = LOSS_CONFIGS[loss]
    print(f"[{LOSS}] Training {config['model_name']}")
    model = NCF(context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 1)
    training_log = []
    stopper = EarlyStopper(patience=PATIENCE)

    for epoch in range(1, NUM_EPOCHS + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=True
        )
        loss = train_epoch(
            model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        ndcg = tune_ndcg(model)
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "tune",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": ndcg,
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(
            f"epoch {epoch:02d} loss={loss:.6f} "
            f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
        )
        if stopper.step(ndcg, model, epoch):
            print("Early stopping")
            break

    stopper.restore(model)
    print(
        f"Best epoch: {stopper.best_epoch} "
        f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
    )

    final_model = NCF(context.n_users, context.n_items, EMBED_DIM, MLP_HIDDEN).to(DEVICE)
    optimizer = torch.optim.AdamW(
        final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rng = np.random.default_rng(SEED + 2)

    for epoch in range(1, stopper.best_epoch + 1):
        started = time.time()
        groups, pos_rows, neg_matrix, neg_len = sample_epoch(
            epoch - 1, fit_only=False
        )
        loss = train_epoch(
            final_model,
            optimizer,
            groups["user_code"].to_numpy().astype(np.int64),
            pos_rows,
            neg_matrix,
            neg_len,
            rng,
        )
        elapsed = time.time() - started
        training_log.append(
            {
                "phase": "refit",
                "epoch": epoch,
                "train_loss": loss,
                "tune_ndcg": float("nan"),
                "ndcg_k": TUNE_NDCG_K,
                "seconds": elapsed,
            }
        )
        print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

    torch.save(
        {
            "state_dict": final_model.state_dict(),
            "config": {
                "n_users": context.n_users,
                "n_items": context.n_items,
                "embed_dim": EMBED_DIM,
                "mlp_hidden": list(MLP_HIDDEN),
                "objective": LOSS,
                "negative_sampling": "per-epoch exposed negatives",
                "negative_count": int(CFG.negative_count),
                "negative_max": int(CFG.negative_count),
                "sampler_epoch_base": 0,
                "max_epochs": NUM_EPOCHS,
                "best_epoch": stopper.best_epoch,
                "epochs": stopper.best_epoch,
                "tune_ndcg": stopper.best_score,
                "ndcg_k": TUNE_NDCG_K,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "seed": SEED,
                "training_data": "labels_train full training pool",
                "training_groups": n_groups,
            },
        },
        config["artifact_dir"] / "model.pt",
    )
    pl.DataFrame(training_log).write_csv(config["artifact_dir"] / "training_log.csv")
    print("Saved model:", config["artifact_dir"] / "model.pt")
    TRAINED_MODELS[LOSS] = final_model


## 5. Results

### Candidate helpers

This helper keeps the candidate format, eligibility policy, history exclusion
and popularity tie-breaking consistent with the other candidate generators.

In [ ]:
def select_top_k(scores, popularity, news_code, k, excluded=None, fallback_order=None):
    if fallback_order is not None:
        return fallback_order[fallback_order < scores.size][:k]
    available = np.ones(scores.size, dtype=bool)
    if excluded is not None:
        available[excluded] = False
    order = np.lexsort((news_code, -popularity, -scores))
    return order[available[order]][:k]


### Score the candidate universe and write Top-500 candidates

Each request uses the static embedding of its `user_code`. Articles not yet
available at the request timestamp are removed. Every article in the request's
`full_history_rows` is excluded, and empty-history requests fall back to the
popularity order. Ties are broken by popularity and then `news_code`.

Because the MLP is not a bilinear dot product, scores are evaluated in bounded
pair chunks rather than one matrix multiply.

In [ ]:
catalogue = pl.DataFrame(
    {
        "article_row": np.arange(1, context.n_items, dtype=np.int32),
        "news_code": context.news_code[1:],
        "available_from": context.available_from[1:],
        "popularity": context.popularity[1:],
    }
).sort(["available_from", "news_code"])

article_rows = catalogue["article_row"].to_numpy().astype(np.int64)
news_sorted = catalogue["news_code"].to_numpy().astype(np.int64)
available_sorted = catalogue["available_from"].to_numpy().astype("datetime64[ns]")
popularity_sorted = catalogue["popularity"].to_numpy().astype(np.float32)

pos_in_sorted = np.full(context.n_items, -1, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(article_rows.size, dtype=np.int64)
global_popularity_order = np.lexsort((news_sorted, -popularity_sorted))

profiles = pl.read_parquet(
    sorted((FEATURES / f"request_profiles_{TARGET_SPLIT}").glob("*.parquet")),
    columns=["impression_uid", "user_code", "timestamp", "full_history_rows"],
).sort("timestamp", maintain_order=True)

impression_uids = profiles["impression_uid"].to_numpy()
target_users = profiles["user_code"].to_numpy().astype(np.int64)
timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
history_rows = profiles["full_history_rows"].to_list()
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
print(f"Scoring requests: {len(timestamps):,}")


In [ ]:
def score_batch(model, user_vectors, item_vectors, max_pairs=SCORE_PAIRS):
    users_t = torch.from_numpy(user_vectors).to(DEVICE)
    items_t = torch.from_numpy(item_vectors).to(DEVICE)
    n_users, n_items = users_t.shape[0], items_t.shape[0]
    items_per_chunk = min(n_items, max_pairs)
    users_per_chunk = max(1, max_pairs // items_per_chunk)
    scores = np.empty((n_users, n_items), dtype=np.float32)
    with torch.no_grad():
        for u0 in range(0, n_users, users_per_chunk):
            u1 = min(u0 + users_per_chunk, n_users)
            user_block = users_t[u0:u1]
            for i0 in range(0, n_items, items_per_chunk):
                i1 = min(i0 + items_per_chunk, n_items)
                rows, cols = u1 - u0, i1 - i0
                scores[u0:u1, i0:i1] = model.score_embeddings(
                    user_block[:, None, :].expand(rows, cols, -1),
                    items_t[i0:i1][None, :, :].expand(rows, cols, -1),
                ).cpu().numpy()
    return scores


In [ ]:
for loss in RUN_LOSSES:
    LOSS = loss
    config = LOSS_CONFIGS[loss]
    model = TRAINED_MODELS[loss]
    print(f"[{LOSS}] Scoring {config['model_name']}")
    model.eval()
    user_emb = model.user_embedding.weight.detach().cpu().numpy()[target_users]
    item_emb = model.item_embedding.weight.detach().cpu().numpy()[article_rows]
    writer = CandidateWriter(config["recommendations_path"])
    started = time.time()
    try:
        for batch_no, (start, end) in enumerate(zip(starts, ends), 1):
            timestamp = timestamps[start]
            n_eligible = int(
                np.searchsorted(available_sorted, timestamp, side="right")
            )
            scores = score_batch(model, user_emb[start:end], item_emb[:n_eligible])
            popularity_batch = popularity_sorted[:n_eligible]
            news_batch = news_sorted[:n_eligible]
            selected_batch = np.empty((end - start, CANDIDATE_CAP), dtype=np.int64)
            for local, profile_row in enumerate(range(start, end)):
                rows = history_rows[profile_row]
                excluded = None
                if rows:
                    excluded = pos_in_sorted[np.asarray(rows, dtype=np.int64)]
                    excluded = excluded[
                        (excluded >= 0) & (excluded < n_eligible)
                    ]
                selected_batch[local] = select_top_k(
                    scores[local],
                    popularity_batch,
                    news_batch,
                    CANDIDATE_CAP,
                    excluded=excluded,
                    fallback_order=None if rows else global_popularity_order,
                )
            writer.add(
                np.repeat(impression_uids[start:end], CANDIDATE_CAP),
                np.tile(cutoff_values, end - start),
                news_batch[selected_batch.reshape(-1)],
            )
            if batch_no % 1000 == 0 or batch_no == len(starts):
                print(
                    f"Scored {batch_no:,}/{len(starts):,} timestamp batches; "
                    f"{writer.rows:,} rows",
                    flush=True,
                )
    finally:
        writer.close()

    print(
        f"Wrote {writer.rows:,} candidate rows in {time.time() - started:.1f}s"
    )
    print("Candidates:", config["recommendations_path"])


### Evaluation and comparison

Uses the shared `CandidateEvaluator` and writes the per-model result and unified
comparison tables.

In [ ]:
from cg_evaluation import CandidateEvaluator

for loss in RUN_LOSSES:
    config = LOSS_CONFIGS[loss]
    evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
    evaluation = evaluator.evaluate_to_csv(
        config["recommendations_path"],
        model_name=config["model_name"],
        m_values=M_VALUES,
        output_dir=RESULTS,
    )
    print(f"[{loss}]")
    print(evaluation.sort("m"))
    print("Results:", config["result_path"])


In [ ]:
model_names = [config["model_name"] for config in LOSS_CONFIGS.values()]
comparison = pl.read_csv(COMPARISON_PATH).filter(pl.col("model").is_in(model_names))
loss_table = (
    comparison.filter(
        pl.col("metric").is_in(
            [
                "recall_pooled",
                "hit_rate",
                "catalogue_coverage",
                "personalisation",
                "long_tail_recall",
                "cold_article_recall",
            ]
        )
    )
    .pivot(on="model", index=["metric", "m"], values="value")
    .sort(["metric", "m"])
)
print(loss_table)


## Notes and limitations

- The user representation is a static `user_code` embedding, not a causal
  request profile. Full `full_history_rows` is used only for exclusion and
  fallback.
- Positives are the clicked articles of each training request (session clicks);
  history-only exposures are not treated as positives.
- Negatives are observed non-clicks from the same slate; up to four are
  redrawn per positive each epoch.
- Training uses a chronological tuning slice, early stopping and a final refit on all training groups.
- The MLP output is a ranking score, not a calibrated click probability.
- Full NCF scoring evaluates the MLP for every request-article pair and is much
  slower than a dot-product model.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.